
### Document: 
**"Digital Transformation of the Healthcare Value Chain: Emergence of Medical Internet of Things (MIoT) may need an Integrated Clinical Environment (ICE) Platform"**

---

## Problem Statement

In today’s digitally evolving healthcare ecosystem, one of the most urgent challenges is reducing preventable medical errors and enabling better clinical decisions in real-time. This research paper highlights how the **Medical Internet of Things (MIoT)** and **Integrated Clinical Environment (ICE)** platforms can help solve that — but all this insight is locked inside a long PDF.

What if we could **chat with this document**?

We aim to build a **smart, RAG-based chatbot** that allows users to ask natural questions and receive context-aware, AI-generated answers directly from the document’s content.

This bot will combine:
- Recursive chunking (to structure the text),
- OpenAI embeddings (to understand meaning),
- ChromaDB (to store and search content),
- LLM reranking (to prioritize the most relevant answers),
- GPT (to generate natural, conversational responses).

Whether the user asks:
- “What role does MIoT play in patient safety?”
- “How does ICE help integrate clinical systems?”
- or “What are the key components of a connected healthcare platform?”

…the chatbot will instantly find the most relevant parts of the document and provide clear, helpful answers.

> In short: We're building an **AI assistant** that transforms a dense research PDF into a **conversational experience** — putting cutting-edge healthcare knowledge at your fingertips.


In [ ]:
#%run $PWD/llm-rag-agents-gateway-labs/Course3/.setup/learner_setup.ipynb

In [ ]:
import os
from dotenv import load_dotenv
import random
import shutil
import httpx
from langchain.vectorstores import Chroma
from langchain_openai import AzureOpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyMuPDFLoader
from langchain.schema import Document
from langchain.chat_models import AzureChatOpenAI
import openai
from langchain.chains import RetrievalQA
from langchain_community.retrievers import BM25Retriever
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)
import warnings

warnings.filterwarnings("ignore")


In [ ]:
# Load variables from .env file
load_dotenv()
# Access variables
api_key = os.getenv('OPENAI_API_KEY')
base_url = os.getenv('OPENAI_BASE_URL')
chat_model_name = os.getenv('CHAT_MODEL_NAME')
embedding_model_name = os.getenv('EMBEDDING_MODEL_NAME')
chat_client = openai.OpenAI(
    api_key=api_key,
    base_url=base_url
)
embeddings_client = openai.OpenAI(
    api_key=api_key,
    base_url=base_url
)

In [ ]:
# This function returns vector embeddings for the provided text chunks.

@retry(wait=wait_random_exponential(min=45, max=120), stop=stop_after_attempt(6))
def get_embeddings(texts_chunk):
    return embeddings_client.embeddings.create(input=texts_chunk, model=embedding_model_name).data

In [ ]:
# Define a Function to Load and Extract Text from PDF
def load_pdf_with_langchain(pdf_path):
 
    # Use LangChain's built-in loader
    loader = PyMuPDFLoader(pdf_path)

    # Load the PDF into LangChain's document format
    documents = loader.load()

    print(f"Successfully loaded {len(documents)} document chunks from the PDF.")
    return documents

In [ ]:
# Path to the uploaded PDF (replace with your actual file path)
pdf_path = "$PWD/llm-rag-agents-gateway-labs/Course3/Module2/Data/HealthcaredocforRAG.pdf"  

# Extract the document chunks
docs = load_pdf_with_langchain(pdf_path)


In [ ]:
# Define a function to chunk documents using RecursiveCharacterTextSplitter.
def chunk_documents(documents, chunk_size=600, chunk_overlap=100):
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(documents)

# Chunk the document
chunked_docs = chunk_documents(docs)
print(f" Total Chunks Created: {len(chunked_docs)}")


In [ ]:
from langchain_openai import OpenAIEmbeddings


embeddings = OpenAIEmbeddings(
    model=embedding_model_name,
    openai_api_key=api_key,
    openai_api_base=base_url
)


In [ ]:
tiktoken_cache_dir = os.path.abspath("$PWD/llm-rag-agents-gateway-labs/Course3/.setup/tiktoken_cache/")
os.environ["TIKTOKEN_CACHE_DIR"] = tiktoken_cache_dir
os.environ["ANONYMIZED_TELEMETRY"]="False"

In [ ]:
# Define a function to create and store embeddings in a local ChromaDB vector store.

@retry(wait=wait_random_exponential(min=45, max=120), stop=stop_after_attempt(6))
def store_embeddings(docs, persist_directory):
    vector_store = Chroma.from_documents(
        docs,
        embedding=embeddings,
        persist_directory=persist_directory
    )
    vector_store.persist()
    return vector_store

# Create and store embeddings
vectorstore = store_embeddings(chunked_docs, persist_directory="/tmp/vectore_store")

In [ ]:
# Define function to retrieve top_k semantically relevant documents from ChromaDB using vector search.
def retrieve_chunks(query,vectorstore, top_k=5):
    results = vectorstore.similarity_search(query, k=top_k*2)  # fetch more to be safe
    unique_results = []
    seen_contents = set()

    for doc in results:
        if doc.page_content not in seen_contents:
            unique_results.append(doc)
            seen_contents.add(doc.page_content)
        if len(unique_results) >= top_k:
            break

    return unique_results

In [ ]:
# Define a function to generate a response based on the top chunks using GPT.
@retry(wait=wait_random_exponential(min=45, max=120), stop=stop_after_attempt(6))
def generate_answer(query, top_chunks, model_name=chat_model_name):

    context = "\n\n".join([doc.page_content for doc in top_chunks])
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    response = chat_client.chat.completions.create(messages=[{"role": "user", "content": prompt}],temperature=0,model=model_name)

    gpt_output = response.choices[0].message.content
    
   
    return gpt_output


In [ ]:
def pdf_chatbot_pipeline(file_path, user_query):
    """
    Full pipeline: Load → Chunk → Embed → Retrieve  → Generate
    """
    raw_docs =  load_pdf_with_langchain(file_path)
    chunks = chunk_documents(raw_docs)
    vectorstore = store_embeddings(chunks, persist_directory="chroma_healthcare.db")
    retrieved = retrieve_chunks(user_query, vectorstore)
    answer = generate_answer(user_query, retrieved)
    return answer

# Example usage
response = pdf_chatbot_pipeline(pdf_path, "How does MIoT improve hospital safety?")
print(response)